# FABLE-5 full highres runner — SLICE-3D + iToBoS

This notebook is meant to live next to the patched `fable5_budget_aware_retrieval` folder inside your `highres` directory. It runs the same engine on:

1. **SLICE-3D / ISIC 2024**: metadata CSV + HDF5 image file.
2. **iToBoS / generic patient dataset**: metadata CSV + image folder.

It writes separate artifacts/results so the two datasets do not overwrite each other:

- `artifacts_slice3d/`, `results_slice3d/`
- `artifacts_itobos/`, `results_itobos/`

The loader now auto-detects common column names, but if iToBoS has unusual names, edit the `ITOBOS_COLS` dict in the setup cell.

In [ ]:
# 0) Install runtime dependencies. Colab usually already has torch, but this is safe.
import sys, subprocess, os, textwrap, json, shutil, glob
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "pyyaml", "pyarrow", "pillow", "h5py", "torch", "torchvision"])
print("Dependencies installed / available.")

In [ ]:
# 1) Mount Drive if running in Colab, then point to your highres folder.
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not in Colab or Drive already mounted:", e)

# EDIT THIS if your folder is somewhere else.
HIGHRES_DIR = Path('/content/drive/MyDrive/highres')
if not HIGHRES_DIR.exists():
    # fallback for local/temporary Colab uploads
    HIGHRES_DIR = Path('/content/highres') if Path('/content/highres').exists() else Path.cwd()
print('HIGHRES_DIR =', HIGHRES_DIR)

# Locate the patched repo folder. If only the zip exists, unzip it.
FABLE_DIR = HIGHRES_DIR / 'fable5_budget_aware_retrieval'
ZIP_CANDIDATES = list(HIGHRES_DIR.glob('fable5_budget_aware_retrieval_PATCHED*.zip')) + list(HIGHRES_DIR.glob('fable5_budget_aware_retrieval*.zip'))
if not FABLE_DIR.exists() and ZIP_CANDIDATES:
    import zipfile
    z = ZIP_CANDIDATES[0]
    print('Unzipping', z)
    with zipfile.ZipFile(z) as f:
        f.extractall(HIGHRES_DIR)
if not FABLE_DIR.exists():
    raise FileNotFoundError(f"Could not find {FABLE_DIR}. Put the patched folder or zip inside HIGHRES_DIR.")
print('FABLE_DIR =', FABLE_DIR)

In [ ]:
# 2) Dataset/path discovery. Autodetects common Kaggle/Drive layouts; edit manually if needed.
import pandas as pd, yaml

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def find_first(root, patterns, max_hits=20):
    root = Path(root)
    hits = []
    for pat in patterns:
        hits += list(root.rglob(pat))[:max_hits]
    return hits[0] if hits else None

# SLICE-3D / ISIC 2024 common paths
SLICE_METADATA_CSV = first_existing([
    HIGHRES_DIR/'isic-2024-challenge/train-metadata.csv',
    HIGHRES_DIR/'slice3d/train-metadata.csv',
    HIGHRES_DIR/'SLICE-3D/train-metadata.csv',
]) or find_first(HIGHRES_DIR, ['*train-metadata*.csv', '*metadata*.csv'])
SLICE_HDF5 = first_existing([
    HIGHRES_DIR/'isic-2024-challenge/train-image.hdf5',
    HIGHRES_DIR/'slice3d/train-image.hdf5',
    HIGHRES_DIR/'SLICE-3D/train-image.hdf5',
]) or find_first(HIGHRES_DIR, ['*train-image*.hdf5', '*.h5', '*.hdf5'])

# iToBoS/common image-folder layout. Edit if autodetect guesses wrong.
ITOBOS_ROOT = first_existing([HIGHRES_DIR/'itobos', HIGHRES_DIR/'iToBoS', HIGHRES_DIR/'ITOBOS']) or HIGHRES_DIR
ITOBOS_METADATA_CSV = find_first(ITOBOS_ROOT, ['*metadata*.csv', '*labels*.csv', '*.csv'])
ITOBOS_IMAGE_DIR = first_existing([
    ITOBOS_ROOT/'images', ITOBOS_ROOT/'Images', ITOBOS_ROOT/'train_images', ITOBOS_ROOT/'img'
]) or ITOBOS_ROOT

# Optional column overrides. Leave None for auto-detect.
SLICE_COLS = dict(id_col='isic_id', patient_col='patient_id', target_col='target', image_path_col=None)
ITOBOS_COLS = dict(id_col=None, patient_col=None, target_col=None, image_path_col=None)

print('SLICE_METADATA_CSV =', SLICE_METADATA_CSV)
print('SLICE_HDF5         =', SLICE_HDF5)
print('ITOBOS_ROOT        =', ITOBOS_ROOT)
print('ITOBOS_METADATA_CSV=', ITOBOS_METADATA_CSV)
print('ITOBOS_IMAGE_DIR   =', ITOBOS_IMAGE_DIR)

for name, csv_path in [('SLICE', SLICE_METADATA_CSV), ('ITOBOS', ITOBOS_METADATA_CSV)]:
    if csv_path and Path(csv_path).exists():
        try:
            df0 = pd.read_csv(csv_path, nrows=3)
            print('
', name, 'columns:', list(df0.columns))
            display(df0.head(3))
        except Exception as e:
            print(name, 'preview failed:', e)

In [ ]:
# 3) Runtime choice. FULL = all patients/lesions; MEDIUM is safer for first pass.
# For final evidence use FULL. If Colab OOMs/timeouts, set RUN_MODE='MEDIUM'.
RUN_MODE = 'FULL'     # 'SMOKE' | 'MEDIUM' | 'FULL'
EMBEDDING_MODEL = 'efficientnet_b0'  # faster than resnet50; change to 'resnet50' if desired
BATCH_SIZE = 96       # lower to 32/48 if GPU memory is tight
SEEDS = [1, 2, 3, 4, 5, 42]

# If you want a bounded first pass even with RUN_MODE='FULL', set e.g. 75000.
MAX_LESIONS_OVERRIDE = None

In [ ]:
# 4) Write dataset-specific configs.
def write_config(name, metadata_csv, hdf5=None, image_dir=None, cols=None):
    if metadata_csv is None or not Path(metadata_csv).exists():
        raise FileNotFoundError(f"{name}: metadata CSV not found: {metadata_csv}")
    if hdf5 is not None and not Path(hdf5).exists():
        raise FileNotFoundError(f"{name}: HDF5 not found: {hdf5}")
    if image_dir is not None and not Path(image_dir).exists():
        raise FileNotFoundError(f"{name}: image_dir not found: {image_dir}")
    cfg = {
        'run_mode': RUN_MODE,
        'paths': {
            'metadata_csv': str(metadata_csv),
            'hdf5': str(hdf5) if hdf5 else None,
            'image_dir': str(image_dir) if image_dir else None,
            'work_dir': str(FABLE_DIR / f'artifacts_{name}'),
            'results_dir': str(FABLE_DIR / f'results_{name}'),
        },
        'data': {
            'id_col': (cols or {}).get('id_col'),
            'patient_col': (cols or {}).get('patient_col'),
            'target_col': (cols or {}).get('target_col'),
            'image_path_col': (cols or {}).get('image_path_col'),
            'positive_values': ['1', 'true', 'yes', 'malignant', 'melanoma', 'cancer', 'positive'],
        },
        'embedding': {'model': EMBEDDING_MODEL, 'use_dino': False, 'batch_size': BATCH_SIZE},
        'subset': {'max_lesions': MAX_LESIONS_OVERRIDE, 'max_patients': None},
        'seeds': SEEDS,
    }
    out = FABLE_DIR / 'configs' / f'fable5_{name}_autogen.yaml'
    with open(out, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print('Wrote', out)
    print(yaml.safe_dump(cfg, sort_keys=False))
    return out

slice_cfg = write_config('slice3d', SLICE_METADATA_CSV, hdf5=SLICE_HDF5, cols=SLICE_COLS)
itobos_cfg = write_config('itobos', ITOBOS_METADATA_CSV, image_dir=ITOBOS_IMAGE_DIR, cols=ITOBOS_COLS)

In [ ]:
# 5) Pipeline runner. It caches embeddings, so re-runs resume after extraction.
def run_cmd(cmd, cwd=FABLE_DIR):
    print('
$ ' + ' '.join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)), cwd=str(cwd))

def run_fable_dataset(cfg_path, error_seed=42):
    run_cmd([sys.executable, 'run_01_build_data.py', '--config', cfg_path])
    run_cmd([sys.executable, 'run_02_extract_embeddings.py', '--config', cfg_path])
    run_cmd([sys.executable, 'run_03_build_features.py', '--config', cfg_path])
    run_cmd([sys.executable, 'run_05_multiseed.py', '--config', cfg_path])
    run_cmd([sys.executable, 'run_06_error_report.py', '--config', cfg_path, '--seed', error_seed])

In [ ]:
# 6) Run SLICE-3D / ISIC 2024 fully.
run_fable_dataset(slice_cfg, error_seed=42)

In [ ]:
# 7) Run iToBoS/generic image-folder dataset fully.
# If this fails at metadata normalization, look at the printed columns above and set ITOBOS_COLS.
run_fable_dataset(itobos_cfg, error_seed=42)

In [ ]:
# 8) Print final reports and compare headline tables.
for name in ['slice3d', 'itobos']:
    res_dir = FABLE_DIR / f'results_{name}'
    print('
' + '='*90)
    print(name.upper(), 'RESULTS:', res_dir)
    report = res_dir / 'readable_report.md'
    if report.exists():
        print(report.read_text()[:6000])
    else:
        print('No readable_report.md found yet.')
    ms = res_dir / 'multiseed_mean_std.csv'
    if ms.exists():
        df = pd.read_csv(ms)
        cols = [c for c in ['method','recall@5_mean','recall@10_mean','mean_rank_first_malignant_mean','normalized_percentile_rank_mean','auroc_mean','pauc_mean'] if c in df.columns]
        display(df[cols].sort_values('recall@5_mean', ascending=False).head(10) if 'recall@5_mean' in cols else df.head())

In [ ]:
# 9) Zip results for download/sharing if needed.
out_zip = FABLE_DIR / 'fable5_results_slice3d_itobos.zip'
if out_zip.exists():
    out_zip.unlink()
shutil.make_archive(str(out_zip.with_suffix('')), 'zip', root_dir=str(FABLE_DIR), base_dir='results_slice3d')
# Append iToBoS results into the same zip.
import zipfile
with zipfile.ZipFile(out_zip, 'a', compression=zipfile.ZIP_DEFLATED) as z:
    base = FABLE_DIR / 'results_itobos'
    if base.exists():
        for p in base.rglob('*'):
            if p.is_file():
                z.write(p, arcname=str(Path('results_itobos') / p.relative_to(base)))
print('Wrote', out_zip)